# 06 - Optimisation des métriques sur GPU (Kaggle)

Ce notebook cherche à obtenir les meilleures métriques possibles sur les vaisseaux, en restant
rigoureux : **tous les choix sont faits sur les 45 volumes de validation**, et les 46 volumes de
test ne servent qu'une fois, à la fin, pour mesurer la configuration retenue.

Leviers testés :

1. un modèle 3D plus fort : DynUNet de MONAI (architecture de nnU-Net), 30,8 M de paramètres,
   entraîné environ 5 h en précision mixte avec une décroissance cosinus du taux d'apprentissage
   calée sur le temps disponible (`configs/kaggle/dynunet.yaml`) ;
2. l'augmentation au test : moyenne des prédictions sur des versions retournées des volumes ;
3. le post-traitement : seuil sur la probabilité des vaisseaux et suppression des petites
   composantes isolées, réglés sur la validation ;
4. les ensembles : moyennes de probabilités du DynUNet, du U-Net 3D et des neuf modèles 2D déjà
   entraînés au notebook 05 (`configs/kaggle/optimisation.yaml`).

Les modèles du notebook 05 sont lus dans la sortie de ce notebook, ajoutée comme donnée d'entrée.
Pendant l'entraînement du DynUNet sur le premier GPU, le second GPU calcule leurs probabilités.

Paramètres Kaggle : deux GPU T4, accès Internet, sortie du notebook
`wavelet-hepatic-vessel-segmentation-gpu` ajoutée en entrée.

In [ ]:
REPO = "https://github.com/HaoudarNouhaila/wavelet-hepatic-vessel-segmentation.git"
BRANCH = "main"
TRAIN_MINUTES = 300   # entraînement du DynUNet (la session Kaggle est limitée à 12 h)
OLD_RUNS = ("unet3d unet2d_s0 unet2d_s1 unet2d_s2 swt_s0 swt_s1 swt_s2 dwt_s0 dwt_s1 dwt_s2")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, glob, shutil, subprocess, time, torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPU :", N_GPU)
!df -h /tmp | cat

## 1. Code, dépendances et données

In [ ]:
PROJECT = "/kaggle/working/project"
if not os.path.exists(PROJECT):
    !git clone -q --depth 1 -b {BRANCH} {REPO} {PROJECT}
%cd {PROJECT}
!git log --oneline -1
!pip install -q monai==1.6.1 PyWavelets==1.8.0 nibabel==5.4.2 SimpleITK==2.5.6
!python -m pytest -q -x 2>&1 | tail -2

In [ ]:
DATA = "/tmp/whvs_data"
os.makedirs(f"{DATA}/raw", exist_ok=True)
if not os.path.exists("data"):
    os.symlink(DATA, "data")
if not os.path.exists("data/raw/Task08_HepaticVessel/labelsTr"):
    !curl -s https://msd-for-monai.s3-us-west-2.amazonaws.com/Task08_HepaticVessel.tar | tar -x -C data/raw --exclude='*/imagesTs/*' --exclude='*/._*'
!ls data/raw/Task08_HepaticVessel/labelsTr | wc -l
!python scripts/run_kaggle.py --steps data --workers 4 --experiments-dir experiments/kaggle_optim

In [ ]:
# Modèles du notebook 05 : copie des checkpoints dans experiments/kaggle/<run>/
copied = 0
for path in glob.glob("/kaggle/input/**/experiments/kaggle/*/best.pt", recursive=True):
    run = os.path.basename(os.path.dirname(path))
    target = f"experiments/kaggle/{run}/best.pt"
    if os.path.isdir(os.path.dirname(target)) and not os.path.exists(target):
        shutil.copy(path, target)
        copied += 1
print(copied, "checkpoints copiés :", sorted(glob.glob("experiments/kaggle/*/best.pt")))

## 2. Probabilités des anciens modèles (second GPU) et entraînement du DynUNet (premier GPU)

In [ ]:
log_old = open("experiments/probs_anciens_modeles.log", "w")
cmd = (f"python scripts/optimize.py --step probs --split val --runs {OLD_RUNS} && "
       f"python scripts/optimize.py --step probs --split test --runs {OLD_RUNS}")
background = None
if N_GPU >= 2:
    background = subprocess.Popen(cmd, shell=True, stdout=log_old, stderr=subprocess.STDOUT,
                                  env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"))
    print("calcul des probabilités lancé sur le GPU 1")
else:
    print("un seul GPU : les probabilités seront calculées après l'entraînement")

In [ ]:
t0 = time.time()
!CUDA_VISIBLE_DEVICES=0 python scripts/train.py --config configs/kaggle/dynunet.yaml --seed 0 --max-minutes {TRAIN_MINUTES} 2>&1 | grep -E "paramètres|époque +[0-9]*[05] |Limite|Terminé|Error|Traceback"
print(f"entraînement : {(time.time() - t0) / 3600:.2f} h")

In [ ]:
if background is not None:
    background.wait()
    log_old.close()
else:
    !{cmd}
!tail -15 experiments/probs_anciens_modeles.log
!python scripts/optimize.py --step probs --split val --runs dynunet
!python scripts/optimize.py --step probs --split test --runs dynunet

## 3. Choix sur la validation

Pour chaque ensemble candidat, Dice moyen des vaisseaux sur les 45 volumes de validation pour
chaque couple (seuil, taille minimale). La configuration retenue est celle du meilleur Dice.

In [ ]:
!python scripts/optimize.py --step search --workers 4
import json, pandas as pd
from IPython.display import Image, display
selection = json.load(open("results/kaggle_optim/metrics/selection.json"))
print("Retenu :", selection["retenu"])
display(pd.DataFrame(selection["meilleur_par_ensemble"]).round(4))

## 4. Évaluation finale sur le test (une seule fois)

In [ ]:
!python scripts/optimize.py --step final --workers 4
!python scripts/optimize.py --step figures
summary = pd.read_csv("results/kaggle_optim/metrics/test_final_summary.csv")
display(summary[summary["class"] == "vessel"].round(4))
for name in ("recherche_validation.png", "comparaison_test.png"):
    display(Image(f"results/kaggle_optim/figures/{name}"))

## 5. Export

In [ ]:
!zip -q -r /kaggle/working/resultats_optimisation.zip results/kaggle_optim experiments/kaggle_optim experiments/probs_anciens_modeles.log -x "*/last.pt"
!ls -lh /kaggle/working/resultats_optimisation.zip